In [2]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

# Load extended dataset
data_path = Path("../data/raw/extended_sales_2013_2025.csv")

df = pd.read_csv(data_path)

print("Extended dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Extended dataset loaded successfully!
Dataset shape: (2374000, 5)


,date,store,item,sales,data_source
0,2013-01-01,1,1,13,original
1,2013-01-01,1,2,33,original
2,2013-01-01,1,3,15,original
3,2013-01-01,1,4,10,original
4,2013-01-01,1,5,11,original


In [3]:
df.tail()

,date,store,item,sales,data_source
2373995,2025-12-31,10,46,57,synthetic
2373996,2025-12-31,10,47,14,synthetic
2373997,2025-12-31,10,48,70,synthetic
2373998,2025-12-31,10,49,32,synthetic
2373999,2025-12-31,10,50,67,synthetic


In [4]:
# Inspect dataset structure

print("DATASET INFORMATION")
print("-" * 50)

df.info()

print("\nCOLUMN DATA TYPES")
print("-" * 50)

print(df.dtypes)

print("\nDATASET DIMENSIONS")
print("-" * 50)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nUNIQUE VALUES")
print("-" * 50)

print("Stores:", df["store"].nunique())
print("Items:", df["item"].nunique())
print("Data Sources:", df["data_source"].unique())

DATASET INFORMATION
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 2374000 entries, 0 to 2373999
Data columns (total 5 columns):
 #   Column       Dtype
---  ------       -----
 0   date         str  
 1   store        int64
 2   item         int64
 3   sales        int64
 4   data_source  str  
dtypes: int64(3), str(2)
memory usage: 90.6 MB

COLUMN DATA TYPES
--------------------------------------------------
date             str
store          int64
item           int64
sales          int64
data_source      str
dtype: object

DATASET DIMENSIONS
--------------------------------------------------
Rows: 2374000
Columns: 5

UNIQUE VALUES
--------------------------------------------------
Stores: 10
Items: 50
Data Sources: <StringArray>
['original', 'synthetic']
Length: 2, dtype: str


In [5]:
# Convert date column to datetime format

df["date"] = pd.to_datetime(df["date"])

# Ensure numerical columns have correct data types

df["store"] = df["store"].astype("int16")
df["item"] = df["item"].astype("int16")
df["sales"] = df["sales"].astype("int32")

# Convert data_source into categorical format

df["data_source"] = df["data_source"].astype("category")

# Verify updated data types

print("Updated Data Types:")
print(df.dtypes)

print("\nDate Conversion Successful!")
print("Start Date:", df["date"].min())
print("End Date:", df["date"].max())

Updated Data Types:
date           datetime64[us]
store                   int16
item                    int16
sales                   int32
data_source          category
dtype: object

Date Conversion Successful!
Start Date: 2013-01-01 00:00:00
End Date: 2025-12-31 00:00:00


In [6]:
# Check missing values in the extended dataset

missing_values = df.isnull().sum()

missing_percentage = (missing_values / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

print("MISSING VALUE ANALYSIS")
print("-" * 50)

display(missing_report)

print("\nTotal Missing Values:", missing_values.sum())

MISSING VALUE ANALYSIS
--------------------------------------------------


,Missing Values,Missing Percentage
date,0,0.0
store,0,0.0
item,0,0.0
sales,0,0.0
data_source,0,0.0



Total Missing Values: 0


In [7]:
# Check duplicate records

duplicate_count = df.duplicated(
    subset=["date", "store", "item"]
).sum()

print("DUPLICATE RECORD ANALYSIS")
print("-" * 50)

print("Total Duplicate Records:", duplicate_count)

if duplicate_count == 0:
    print("No duplicate records found!")
else:
    print("Duplicate records detected.")

    duplicates = df[
        df.duplicated(
            subset=["date", "store", "item"],
            keep=False
        )
    ]

    display(duplicates.head(10))

DUPLICATE RECORD ANALYSIS
--------------------------------------------------
Total Duplicate Records: 0
No duplicate records found!


In [8]:
# Validate sales values

print("SALES DATA VALIDATION")
print("-" * 50)

print("Minimum Sales:", df["sales"].min())
print("Maximum Sales:", df["sales"].max())
print("Negative Sales:", (df["sales"] < 0).sum())
print("Zero Sales:", (df["sales"] == 0).sum())

print("\nDATA SOURCE DISTRIBUTION")
print("-" * 50)

source_counts = df["data_source"].value_counts()

display(source_counts.to_frame(name="Record Count"))

print("\nDATA SOURCE PERCENTAGE")
display(
    (df["data_source"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame(name="Percentage")
)

SALES DATA VALIDATION
--------------------------------------------------
Minimum Sales: 0
Maximum Sales: 240
Negative Sales: 0
Zero Sales: 44

DATA SOURCE DISTRIBUTION
--------------------------------------------------


,Record Count
data_source,
synthetic,1461000
original,913000



DATA SOURCE PERCENTAGE


,Percentage
data_source,
synthetic,61.54
original,38.46


In [9]:
# Analyze year-wise record distribution

df["year"] = df["date"].dt.year

yearly_records = (
    df.groupby(["year", "data_source"], observed=True)
      .size()
      .unstack(fill_value=0)
)

print("YEAR-WISE DATA DISTRIBUTION")
print("-" * 60)

display(yearly_records)

print("\nTOTAL RECORDS PER YEAR")
display(df.groupby("year").size().to_frame("Total Records"))

YEAR-WISE DATA DISTRIBUTION
------------------------------------------------------------


data_source,original,synthetic
year,,
2013,182500,0
2014,182500,0
2015,182500,0
2016,183000,0
2017,182500,0
2018,0,182500
2019,0,182500
2020,0,183000
2021,0,182500



TOTAL RECORDS PER YEAR


,Total Records
year,
2013,182500
2014,182500
2015,182500
2016,183000
2017,182500
2018,182500
2019,182500
2020,183000
2021,182500


In [10]:
# Validate yearly coverage and data source consistency

yearly_counts = df.groupby("year").size()

expected_years = list(range(2013, 2026))

expected_counts = {
    year: (366 if year in [2016, 2020, 2024] else 365) * 500
    for year in expected_years
}

print("YEARLY COVERAGE VALIDATION")
print("-" * 50)

for year in expected_years:
    actual = yearly_counts.get(year, 0)
    expected = expected_counts[year]

    status = "PASS" if actual == expected else "FAIL"

    print(f"{year}: {actual:,} / {expected:,} - {status}")

print("\nSOURCE CONSISTENCY")
print("-" * 50)

print("Original records after 2017:",
      len(df[(df["year"] > 2017) & (df["data_source"] == "original")]))

print("Synthetic records before 2018:",
      len(df[(df["year"] < 2018) & (df["data_source"] == "synthetic")]))

YEARLY COVERAGE VALIDATION
--------------------------------------------------
2013: 182,500 / 182,500 - PASS
2014: 182,500 / 182,500 - PASS
2015: 182,500 / 182,500 - PASS
2016: 183,000 / 183,000 - PASS
2017: 182,500 / 182,500 - PASS
2018: 182,500 / 182,500 - PASS
2019: 182,500 / 182,500 - PASS
2020: 183,000 / 183,000 - PASS
2021: 182,500 / 182,500 - PASS
2022: 182,500 / 182,500 - PASS
2023: 182,500 / 182,500 - PASS
2024: 183,000 / 183,000 - PASS
2025: 182,500 / 182,500 - PASS

SOURCE CONSISTENCY
--------------------------------------------------
Original records after 2017: 0
Synthetic records before 2018: 0


In [11]:
# Final data cleaning

# Remove temporary year column
df.drop(columns=["year"], inplace=True)

# Sort records chronologically
df = df.sort_values(
    by=["date", "store", "item"]
).reset_index(drop=True)

# Verify final structure
print("FINAL DATASET SUMMARY")
print("-" * 50)

print("Shape:", df.shape)
print("Date Range:", df["date"].min(), "to", df["date"].max())
print("Stores:", df["store"].nunique())
print("Items:", df["item"].nunique())
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Records:", df.duplicated(
    subset=["date", "store", "item"]
).sum())

display(df.head())

FINAL DATASET SUMMARY
--------------------------------------------------
Shape: (2374000, 5)
Date Range: 2013-01-01 00:00:00 to 2025-12-31 00:00:00
Stores: 10


C:\Users\aditi\AppData\Local\Temp\ipykernel_35360\4240880222.py:4: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["year"], inplace=True)


Items: 50
Missing Values: 0
Duplicate Records: 0


,date,store,item,sales,data_source
0,2013-01-01,1,1,13,original
1,2013-01-01,1,2,33,original
2,2013-01-01,1,3,15,original
3,2013-01-01,1,4,10,original
4,2013-01-01,1,5,11,original


In [12]:
# Save cleaned extended dataset

from pathlib import Path

output_path = Path("../data/processed/cleaned_extended_sales.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("Cleaned extended dataset saved successfully!")
print("File:", output_path)
print("Total Records:", len(df))

Cleaned extended dataset saved successfully!
File: ..\data\processed\cleaned_extended_sales.csv
Total Records: 2374000
